## Build a Question Answer Database over a Graph Database

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_URI = os.getenv('NEO4J_URI')

In [2]:
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = "true"
os.environ['LANGCHAIN_PROJECT'] = "Langchain with GraphDB"

In [3]:
os.environ['NEO4J_USERNAME'] = NEO4J_USERNAME
os.environ['NEO4J_PASSWORD'] = NEO4J_PASSWORD
os.environ['NEO4J_URI'] = NEO4J_URI

In [4]:
from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph(url= NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)
graph

c:\Users\singh\OneDrive\Desktop\Generative AI\GenAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
## Dataset Movies
movie_query = """
LOAD CSV WITH HEADERS FROM 
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/refs/heads/main/movies/movies_small.csv'
AS row

MERGE (m:Movie {id: toInteger(row.movieId)})
SET m.title = row.title,
    m.released = toInteger(row.released),
    m.tagline = row.tagline,
    m.imdbRating = toFloat(row.imdbRating)

WITH m, row

// Directors
FOREACH (director IN 
    CASE WHEN row.director IS NOT NULL AND row.director <> "" 
    THEN split(row.director, '|') ELSE [] END |

    MERGE (p:Person {name: toLower(trim(director))})
    MERGE (p)-[:DIRECTED]->(m)
)

// Actors
FOREACH (actor IN 
    CASE WHEN row.actors IS NOT NULL AND row.actors <> "" 
    THEN split(row.actors, '|') ELSE [] END |

    MERGE (p:Person {name: toLower(trim(actor))})
    MERGE (p)-[:ACTED_IN]->(m)
)

// Genres
FOREACH (genre IN 
    CASE WHEN row.genres IS NOT NULL AND row.genres <> "" 
    THEN split(row.genres, '|') ELSE [] END |

    MERGE (g:Genre {name: toLower(trim(genre))})
    MERGE (m)-[:IN_GENRE]->(g)
)
"""

movie_query

'\nLOAD CSV WITH HEADERS FROM \n\'https://raw.githubusercontent.com/tomasonjo/blog-datasets/refs/heads/main/movies/movies_small.csv\'\nAS row\n\nMERGE (m:Movie {id: toInteger(row.movieId)})\nSET m.title = row.title,\n    m.released = toInteger(row.released),\n    m.tagline = row.tagline,\n    m.imdbRating = toFloat(row.imdbRating)\n\nWITH m, row\n\n// Directors\nFOREACH (director IN \n    CASE WHEN row.director IS NOT NULL AND row.director <> "" \n    THEN split(row.director, \'|\') ELSE [] END |\n\n    MERGE (p:Person {name: toLower(trim(director))})\n    MERGE (p)-[:DIRECTED]->(m)\n)\n\n// Actors\nFOREACH (actor IN \n    CASE WHEN row.actors IS NOT NULL AND row.actors <> "" \n    THEN split(row.actors, \'|\') ELSE [] END |\n\n    MERGE (p:Person {name: toLower(trim(actor))})\n    MERGE (p)-[:ACTED_IN]->(m)\n)\n\n// Genres\nFOREACH (genre IN \n    CASE WHEN row.genres IS NOT NULL AND row.genres <> "" \n    THEN split(row.genres, \'|\') ELSE [] END |\n\n    MERGE (g:Genre {name: toLowe

In [6]:
graph.query(query=movie_query)

[]

In [7]:
graph.refresh_schema()
print(graph.schema)

Node properties:
CEO {POB: STRING, YOB: INTEGER, name: STRING}
Company {name: STRING}
AI_Engineer {POB: STRING, YOB: STRING, name: STRING}
Country {name: STRING}
Person {name: STRING, born: INTEGER}
Movie {released: INTEGER, title: STRING, id: INTEGER, imdbRating: FLOAT}
User {name: STRING, city: STRING, userId: INTEGER, age: INTEGER}
Post {content: STRING, postId: INTEGER, timestamp: DATE_TIME}
Genre {name: STRING}
Relationship properties:

The relationships:
(:AI_Engineer)-[:LIVES_IN]->(:Country)
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)
(:User)-[:POSTED]->(:Post)


In [8]:
model_api = os.getenv('NVIDIA_MODEL_API')
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(
  model="openai/gpt-oss-20b",
  api_key= model_api, 
  temperature=1,
  top_p=1,
  max_tokens=4096,
)
llm

C:\Users\singh\AppData\Local\Temp\ipykernel_2152\811944973.py:4: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(


ChatNVIDIA(output_version=None, profile={}, base_url='https://integrate.api.nvidia.com/v1', model='openai/gpt-oss-20b', temperature=1.0, max_tokens=4096, top_p=1.0, default_headers={}, model_kwargs={})

In [9]:
from langchain_neo4j import GraphCypherQAChain
chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, allow_dangerous_requests=True, verbose=True)
chain 

GraphCypherQAChain(verbose=True, graph=<langchain_neo4j.graphs.neo4j_graph.Neo4jGraph object at 0x0000018D80A69F90>, cypher_generation_chain=PromptTemplate(input_variables=['examples', 'question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nExamples (optional):\n{examples}\n\nThe question is:\n{question}')
| RunnableBinding(bound=ChatNVIDIA(output_version=None, profile={}, base_url='https://integrate.api.nvidia.com/v1', model='openai/gpt-oss-20b', temperature=1.0, max_tokens=4096, top_p=1.0, 

In [10]:
response = chain.invoke({"query": "Who is the Director of the movie Casino"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie)
WHERE m.title = 'Casino'
RETURN p.name AS director;
Full Context:
[{'director': 'martin scorsese'}]

> Finished chain.


{'query': 'Who is the Director of the movie Casino',
 'result': 'Martin Scorsese is the director of the movie Casino.'}

In [11]:
response = chain.invoke({"query": "Who were the actors in the movie Casino"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title: 'Casino'})
RETURN p.name;
Full Context:
[{'p.name': 'robert de niro'}, {'p.name': 'joe pesci'}, {'p.name': 'sharon stone'}, {'p.name': 'james woods'}]

> Finished chain.


{'query': 'Who were the actors in the movie Casino',
 'result': 'Robert De\u202fNiro, Joe\u202fPesci, Sharon\u202fStone, James\u202fWoods.'}